# 02 — Baseline TF-IDF + LogReg (ponderado)

TF-IDF (50k features, bigramas) + Regressão Logística com `class_weight='balanced'`,
**sempre sobre `VOTO_LIMPO`**. Nunca sobre SUMARIO.

Avaliação em duas modalidades:
- **K-Fold 5×** estratificado — visão média + IC 95%.
- **Hold-out temporal** — treino ≤ 2022, val = 2023, teste = 2024.

**Saída:** `resultados/metricas_baseline.json` e figuras em `resultados/figuras/`.


## 1. Setup


In [ ]:
import os, sys, subprocess
REPO_DIR = os.environ.get('REPO_DIR', '/content/deep-acordao-tcu2')
REPO_URL = 'https://github.com/bsousa7/deep-acordao-tcu2.git'
BRANCH = os.environ.get('BRANCH', 'claude/deep-acordao-tcu-refactor-yyjfr3')
if not os.path.isdir(os.path.join(REPO_DIR, 'src')):
    subprocess.run(['git', 'clone', REPO_URL, '--branch', BRANCH, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


In [ ]:
%pip -q install pandas pyarrow scikit-learn scipy nltk matplotlib seaborn


## 2. Carrega splits do notebook 01


In [ ]:
from pathlib import Path
import logging, pandas as pd, numpy as np
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

BASE = Path(REPO_DIR)

# Busca os parquets gerados pelo notebook 01 — primeiro no Google Drive
# (persistência entre sessões), com fallback para o clone local (útil se
# 01 e 02 rodarem na mesma sessão sem Drive montado).
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive') / 'deep-acordao-tcu2'
except Exception as _e:
    DRIVE_ROOT = None
    print(f'Google Drive indisponível (fora do Colab?): {_e}')

_drive_interim = (DRIVE_ROOT / 'data' / 'interim') if DRIVE_ROOT else None
PERSIST_BASE = DRIVE_ROOT if (_drive_interim and _drive_interim.exists()) else BASE

DATA_INTERIM = PERSIST_BASE / 'data' / 'interim'
DATA_PROCESSED = PERSIST_BASE / 'data' / 'processed'
RESULTADOS = PERSIST_BASE / 'resultados'
FIGURAS = RESULTADOS / 'figuras'; FIGURAS.mkdir(parents=True, exist_ok=True)

print(f'Lendo dados de: {PERSIST_BASE}')
df_full = pd.read_parquet(DATA_INTERIM / 'acordaos_rotulados.parquet')
train_df = pd.read_parquet(DATA_PROCESSED / 'train.parquet')
val_df   = pd.read_parquet(DATA_PROCESSED / 'val.parquet')
test_df  = pd.read_parquet(DATA_PROCESSED / 'test.parquet')
print(f'full={len(df_full)}  treino={len(train_df)}  val={len(val_df)}  teste={len(test_df)}')


## 3. Limpeza TF-IDF (sobre VOTO_LIMPO)


In [ ]:
from src.preprocessamento.limpeza import aplicar_limpeza

for split in [df_full, train_df, val_df, test_df]:
    split['VOTO_TFIDF'] = aplicar_limpeza(split, campo='VOTO_LIMPO', modo='tfidf')
print('OK — coluna VOTO_TFIDF gerada em todos os splits.')


## 4. K-Fold 5× com `class_weight='balanced'`


In [ ]:
from src.modelos.baseline import treinar_kfold
from src.modelos.pesos import pesos_balanceados

print('Pesos balanceados calculados no corpus completo (referência):')
print({k: round(v, 3) for k, v in pesos_balanceados(df_full['LABEL']).items()})

res_kfold_balanced = treinar_kfold(df_full['VOTO_TFIDF'], df_full['LABEL'], class_weight='balanced')
print(f"\nF1-macro = {res_kfold_balanced['mean_f1']:.4f}  IC95={res_kfold_balanced['ci_95']}")


## 5. Contraste — sem ponderação (para demonstrar o efeito)


In [ ]:
res_kfold_nada = treinar_kfold(df_full['VOTO_TFIDF'], df_full['LABEL'], class_weight=None)
print(f"F1-macro (SEM pesos) = {res_kfold_nada['mean_f1']:.4f}")
print('F1 por classe SEM pesos:', {k: round(v, 4) for k, v in res_kfold_nada['per_class_f1'].items()})
print('F1 por classe COM balanced:', {k: round(v, 4) for k, v in res_kfold_balanced['per_class_f1'].items()})


## 6. Hold-out temporal — treino ≤ 2022, val = 2023, teste = 2024


In [ ]:
from src.modelos.baseline import treinar_holdout
from src.avaliacao.metricas import relatorio, plotar_matriz_confusao, resumo_metricas

pipe, metricas_holdout = treinar_holdout(
    train_df['VOTO_TFIDF'], train_df['LABEL'],
    test_df['VOTO_TFIDF'], test_df['LABEL'],
    class_weight='balanced',
)
y_pred = pipe.predict(test_df['VOTO_TFIDF'])
_ = relatorio(test_df['LABEL'], y_pred, titulo='Baseline hold-out temporal')
plotar_matriz_confusao(test_df['LABEL'], y_pred, titulo='baseline_holdout', output_dir=FIGURAS)


## 7. Persistência


In [ ]:
from src.avaliacao.metricas import salvar_json

saida = {
    'representacao': 'TF-IDF 50k bigramas + LogReg',
    'feature': 'VOTO_LIMPO (sem SUMARIO, sem dispositivo, veredito residual mascarado)',
    'kfold_5x_balanced': {
        'mean_f1': res_kfold_balanced['mean_f1'],
        'std_f1':  res_kfold_balanced['std_f1'],
        'ci_95':   list(res_kfold_balanced['ci_95']),
        'mean_acc': res_kfold_balanced['mean_acc'],
        'per_class_f1': res_kfold_balanced['per_class_f1'],
        'confusion_matrix': res_kfold_balanced['confusion_matrix'],
    },
    'kfold_5x_sem_pesos': {
        'mean_f1': res_kfold_nada['mean_f1'],
        'per_class_f1': res_kfold_nada['per_class_f1'],
    },
    'holdout_temporal_2024': metricas_holdout,
}
salvar_json(saida, RESULTADOS / 'metricas_baseline.json')
